# Notebook 3 — Spectral Indices


**Objectives**
- Know the formulas for NDVI, NDWI, MNDWI, NDBI and what each is sensitive to
- Compute indices **server-side** (GEE) at scale
- Compute the same indices **locally** on a downloaded raster (numpy/rasterio)
- Understand when to use which path, and confirm they agree

In [ ]:
import ee, geemap
ee.Initialize(project="operating-rush-505912-s8")

AOI = ee.Geometry.Rectangle([36.20, -0.83, 36.45, -0.65])  # Lake Naivasha
START, END = "2023-01-01", "2023-03-01"

def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask).divide(10000).copyProperties(image, image.propertyNames())

s2_composite = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterBounds(AOI).filterDate(START, END)
                .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
                .map(mask_s2_clouds)
                .median().clip(AOI))


## Formulas

| Index | Formula | Bands (S2) | Senses |
|---|---|---|---|
| **NDVI** | $(NIR - Red) / (NIR + Red)$ | $(B8 - B4)/(B8 + B4)$ | Vegetation vigour |
| **NDWI** | $(Green - NIR) / (Green + NIR)$ | $(B3 - B8)/(B3 + B8)$ | Open water / vegetation moisture (McFeeters) |
| **MNDWI** | $(Green - SWIR1) / (Green + SWIR1)$ | $(B3 - B11)/(B3 + B11)$ | Open water, more robust than NDWI in built-up/turbid scenes |
| **NDBI** | $(SWIR1 - NIR) / (SWIR1 + NIR)$ | $(B11 - B8)/(B11 + B8)$ | Built-up / impervious surfaces |

MNDWI swaps NDWI's NIR band for SWIR1 specifically because built-up surfaces are bright in NIR (confusing NDWI) but dark in SWIR — this is why MNDWI is the index used for the SAR-vs-optical water cross-check in Notebook 5.

## Server-side path
GEE's `.normalizedDifference([b1, b2])` is a shortcut for `(b1-b2)/(b1+b2)`. Use `.expression()` when the formula isn't a plain normalized difference.

In [ ]:
ndvi  = s2_composite.normalizedDifference(["B8", "B4"]).rename("NDVI")
ndwi  = s2_composite.normalizedDifference(["B3", "B8"]).rename("NDWI")
mndwi = s2_composite.normalizedDifference(["B3", "B11"]).rename("MNDWI")

indices_server = ndvi.addBands([ndwi, mndwi])
print(indices_server.bandNames().getInfo())

In [ ]:
Map = geemap.Map(center=[-0.74, 36.33], zoom=11)
Map.addLayer(s2_composite, {"bands": ["B4","B3","B2"], "min": 0, "max": 0.3}, "S2 true color")
Map.addLayer(ndvi,  {"min": -0.2, "max": 0.8, "palette": ["brown","yellow","green"]}, "NDVI (server)")
Map.addLayer(mndwi, {"min": -0.5, "max": 0.5, "palette": ["white","blue"]}, "MNDWI (server)")
Map

## Local raster path
Download the composite once, then compute indices with plain numpy. Useful when: you need indices outside GEE's compute model (custom filters, ML pre-processing), or you're working from imagery that never touched GEE (drone orthomosaics, local archives).

In [ ]:
import numpy as np, rasterio

# Download the 4 bands we need, at native 10 m
geemap.ee_export_image(
    s2_composite.select(["B3","B4","B8","B11"]),
    filename="naivasha_bands.tif",
    scale=10, region=AOI, crs="EPSG:32637", file_per_band=False
)

with rasterio.open("naivasha_bands.tif") as src:
    green, red, nir, swir1 = [src.read(i).astype("float32") for i in range(1, 5)]
    profile = src.profile

def norm_diff(a, b):
    with np.errstate(divide="ignore", invalid="ignore"):
        out = (a - b) / (a + b)
    return np.where(np.isfinite(out), out, np.nan)

ndvi_local  = norm_diff(nir, red)
ndwi_local  = norm_diff(green, nir)
mndwi_local = norm_diff(green, swir1)

print("NDVI local  — min/max/mean:", np.nanmin(ndvi_local), np.nanmax(ndvi_local), np.nanmean(ndvi_local))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, arr, title, cmap in zip(
    axes, [ndvi_local, ndwi_local, mndwi_local],
    ["NDVI (local)", "NDWI (local)", "MNDWI (local)"], ["RdYlGn", "Blues", "Blues"]
):
    im = ax.imshow(arr, cmap=cmap, vmin=-0.5, vmax=0.8)
    ax.set_title(title); ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

## Server vs. local — do they agree?
Sample the same pixel location from both paths. Small floating-point differences (~1e-6) are expected from resampling/interpolation order; large differences signal a bug (wrong band order, un-applied scale factor, CRS mismatch).

In [ ]:
sample_point = ee.Geometry.Point([36.33, -0.74])

server_vals = indices_server.reduceRegion(
    reducer=ee.Reducer.first(), geometry=sample_point, scale=10
).getInfo()

row, col = src.index(36.33, -0.74)
local_vals = {
    "NDVI": float(ndvi_local[row, col]),
    "NDWI": float(ndwi_local[row, col]),
    "MNDWI": float(mndwi_local[row, col]),
}

print("Server:", server_vals)
print("Local :", local_vals)
print("\nMax abs difference:",
      max(abs(server_vals[k] - local_vals[k]) for k in local_vals))

**When to use which path:**
| | Server-side (GEE) | Local (numpy/rasterio) |
|---|---|---|
| Best for | Large-area, repeatable, production pipelines | Small AOIs, custom algorithms, non-GEE-native data |
| Scaling | Handles country/continent scale without downloading | Limited by local RAM/disk |
| Iteration speed | Network round-trip per change | Instant once data is local |
| Reproducibility | Depends on collection versioning | Fully deterministic, file-based |

## NDBI addition
Built-up index — useful as a *negative control* around the lake shoreline (distinguishing shoreline settlement from water/vegetation in Notebook 5's classification checks).

In [ ]:
ndbi = s2_composite.normalizedDifference(["B11", "B8"]).rename("NDBI")
Map.addLayer(ndbi, {"min": -0.3, "max": 0.3, "palette": ["blue","white","red"]}, "NDBI (server)")

ndbi_local = norm_diff(swir1, nir)
Map

## Exercise 3.1 — Naivasha indices
Using the **dry-season composite** you exported in Exercise 2.1:
1. Compute NDVI, MNDWI, and NDBI server-side
2. Download the same bands and reproduce all three locally
3. Confirm agreement at 3 sample points of your choice (lake, farmland, built-up area near Naivasha town)
4. Note which index best isolates the lake shoreline in the dry season vs. the wet-season result from Notebook 3's main example

In [ ]:
# --- Exercise 3.1 scaffold ---
# TODO 1: load your Exercise 2.1 dry-season composite (asset or re-filter+composite)
# TODO 2: compute ndvi_dry, mndwi_dry, ndbi_dry server-side
# TODO 3: download bands, compute local equivalents
# TODO 4: sample 3 points and compare server vs local
